# 01 – Data audit 

Goal: find every data problem in the four raw files, count how many rows it affects,
and decide whether to **fix, flag or drop** it, with a reason.
Findings are collected in an issue log and summarized in `findings.md`.

Raw files are never edited; every change is made in code so it can be reproduced
and later applied inside the cleaning transform.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
FILES = {
    "applications": "applications.csv",
    "bureau": "credit_bureau.csv",
    "servicing": "loan_servicing.csv",
    "holdout": "holdout_applications.csv",
}

def load_raw(raw_dir=RAW):
    """Read each raw file; force id to text so ids like '0123456789' keep their leading zero."""
    return {name: pd.read_csv(raw_dir / fname, dtype={"id": str}, low_memory=False)
            for name, fname in FILES.items()}

data = load_raw()
apps, bureau, serv, hold = (data[k] for k in ["applications", "bureau", "servicing", "holdout"])

### Issue log
Every confirmed problem is recorded with its file, column, number of affected rows,
the action taken (fix / flag / drop) and the reason. At the end of the audit this
becomes the issue table in `findings.md`.

In [19]:
ISSUES = {}
ISSUES_PATH = Path("../reports/audit_issues.csv")

def log_issue(file, column, issue, rows, action, reason):
    """Record one data problem. Re-running replaces the entry instead of duplicating it."""
    ISSUES[(file, column, issue)] = {"file": file, "column": column, "issue": issue,
                                     "rows": int(rows), "action": action, "reason": reason}

def save_issues():
    """Write the issue log to reports/ so it survives restarts and goes into findings.md."""
    table = pd.DataFrame(ISSUES.values())
    table.to_csv(ISSUES_PATH, index=False)
    return table

### Duplicates in each file
Two kinds of duplicates are counted:
- **Exact duplicates:** a whole row repeated.
- **Same id, different values:** the same loan recorded more than once with
  conflicting details, which is harder to resolve.

Duplicates would make a loan count twice in training and in the default rate.

In [3]:
def duplicate_report(df):
    """Two kinds of duplicates: identical copies, and same id with different values."""
    exact = df.duplicated().sum()                    # whole row repeated
    conflicting = df.drop_duplicates()["id"].duplicated().sum()  # same id, values differ
    return {"rows": len(df), "unique_ids": df["id"].nunique(),
            "exact_duplicates": int(exact), "same_id_different_values": int(conflicting)}

for name, df in data.items():
    print(name, duplicate_report(df))

applications {'rows': 30600, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 600}
bureau {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
servicing {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
holdout {'rows': 5000, 'unique_ids': 5000, 'exact_duplicates': 0, 'same_id_different_values': 0}


### What differs between conflicting copies?
For a few repeated ids, only the columns where the copies disagree are shown.
Knowing *what* differs tells us whether the copies are the same loan with
formatting differences, or genuinely different records.

In [4]:
def show_conflicts(df, n=3):
    """For a few ids that repeat with different values, show only the columns that differ."""
    deduped = df.drop_duplicates()
    ids = deduped.loc[deduped["id"].duplicated(), "id"].unique()[:n]
    for loan_id in ids:
        rows = deduped[deduped["id"] == loan_id]
        diff_cols = rows.columns[rows.nunique(dropna=False) > 1].tolist()
        print(f"\nid {loan_id} differs in: {diff_cols}")
        display(rows[["id"] + diff_cols])

show_conflicts(apps)


id 8acbab3c06 differs in: ['emp_title', 'dti']


,id,emp_title,dti
108,8acbab3c06,Hillstone,15.53
1199,8acbab3c06,HILLSTONE,15.54



id ddefd45498 differs in: ['emp_title', 'dti']


,id,emp_title,dti
429,ddefd45498,Safety Manager,15.36
1214,ddefd45498,SAFETY MANAGER,15.37



id 01edeb5c98 differs in: ['emp_title', 'dti']


,id,emp_title,dti
1083,01edeb5c98,Legal Negotiator,14.49
1506,01edeb5c98,LEGAL NEGOTIATOR,14.50


### Which columns differ, across all repeated ids?
Before deciding, we check that the pattern seen in the examples holds for every
repeated id. If an important field such as `defaulted` or `loan_amnt` ever
differed, the copies could not simply be merged.

In [7]:
def conflict_summary(df):
    """For every id that repeats with different values, count which columns differ."""
    deduped = df.drop_duplicates()
    dup = deduped[deduped["id"].duplicated(keep=False)]   # all copies of repeated ids
    counts = {}
    for loan_id, rows in dup.groupby("id"):
        for col in rows.columns[rows.nunique(dropna=False) > 1]:
            counts[col] = counts.get(col, 0) + 1
    print("Repeated ids:", dup["id"].nunique())
    return pd.Series(counts, dtype=int).sort_values(ascending=False)

conflict_summary(apps)

Repeated ids: 600


dti          600
emp_title    528
dtype: int64

In [8]:
dup = apps.drop_duplicates()
dup = dup[dup["id"].duplicated(keep=False)]
g = dup.groupby("id")

case_only = g["emp_title"].agg(lambda s: s.str.lower().nunique(dropna=False) == 1)
print("emp_title differs only by case:", case_only.all())
print("Largest dti difference:", (g["dti"].max() - g["dti"].min()).max())

emp_title differs only by case: True
Largest dti difference: 0.010000000000005116


### Decision: drop duplicate ids (keep first)
600 ids appear twice in `applications.csv`. The copies differ only in `emp_title`
capitalization (528 pairs) and `dti` by at most 0.01 (600 pairs); the label and
amounts never differ. They are the same loan entered twice, so the second copy is
dropped. Keeping both would double-count these loans.
This leaves 30,000 applications, matching the bureau and servicing files.

In [20]:
before = len(apps)
apps = apps.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"applications: {before} → {len(apps)} rows")

log_issue(
    file="applications", column="id",
    issue="600 ids appear twice; copies differ only in emp_title capitalization (528) and dti by ≤0.01 (600)",
    rows=600,
    action="drop duplicate (keep first)",
    reason="Same loan entered twice; keeping both would double-count it. "
           "Differences are formatting/rounding, and the label and amounts never differ",
)

applications: 30000 → 30000 rows


### Do the three files cover the same loans?
Compares the sets of ids between applications, bureau and servicing. A loan
missing from one file would lose its bureau data or its servicing check when joined.

In [5]:
def coverage(df_a, df_b, name_a, name_b):
    """Ids present in one file but not the other."""
    a, b = set(df_a["id"]), set(df_b["id"])
    print(f"{name_a} ids missing from {name_b}: {len(a - b)}")
    print(f"{name_b} ids missing from {name_a}: {len(b - a)}")

coverage(apps, bureau, "applications", "bureau")
coverage(apps, serv, "applications", "servicing")
coverage(bureau, serv, "bureau", "servicing")

applications ids missing from bureau: 0
bureau ids missing from applications: 0
applications ids missing from servicing: 0
servicing ids missing from applications: 0
bureau ids missing from servicing: 0
servicing ids missing from bureau: 0


### Does the holdout overlap with training?
Holdout loans are from 2016 and should not appear in the training files.
Any overlap would put the same loan in both training and final scoring.

In [6]:
overlap = set(hold["id"]) & set(apps["id"])
print("Holdout ids also in applications:", len(overlap))

Holdout ids also in applications: 0


## A1.2 Formats

### Date formats
Each date value is reduced to its "shape" (letters → `A`, digits → `9`), so
`Dec-2013` becomes `AAA-9999` and `2013-12-01` becomes `9999-99-99`. More than one
shape in a column means mixed date formats, which can be parsed incorrectly or
silently, for example by swapping day and month.

In [10]:
def pattern_counts(s):
    """Reduce each value to its 'shape': letters become A, digits become 9."""
    shapes = (s.dropna().astype(str)
              .str.replace(r"[A-Za-z]", "A", regex=True)
              .str.replace(r"\d", "9", regex=True))
    return shapes.value_counts()

DATE_COLS = {
    "applications": ["issue_d"],
    "bureau": ["earliest_cr_line"],
    "servicing": ["last_pymnt_d", "last_credit_pull_d"],
    "holdout": ["issue_d", "earliest_cr_line"],
}

for name, cols in DATE_COLS.items():
    for col in cols:
        print(f"\n--- {name}.{col} ---")
        print(pattern_counts(data[name][col] if name != "applications" else apps[col]))


--- applications.issue_d ---
issue_d
AAA-9999      28800
9999-99-99      615
99/9999         585
Name: count, dtype: int64

--- bureau.earliest_cr_line ---
earliest_cr_line
AAA-9999      28800
9999-99-99      607
99/9999         593
Name: count, dtype: int64

--- servicing.last_pymnt_d ---
last_pymnt_d
AAA-9999    29979
Name: count, dtype: int64

--- servicing.last_credit_pull_d ---
last_credit_pull_d
AAA-9999    29999
Name: count, dtype: int64

--- holdout.issue_d ---
issue_d
AAA-9999    5000
Name: count, dtype: int64

--- holdout.earliest_cr_line ---
earliest_cr_line
AAA-9999    5000
Name: count, dtype: int64


### Case and spacing variants in categories
Counts distinct values before and after removing surrounding spaces and lowercasing.
If normalizing reduces the count, the same category is written in more than one way
(e.g. `RENT` vs `rent `), which would split one group into several.
The holdout is checked too, because it may contain variants that training does not.

In [11]:
TEXT_COLS = ["term", "grade", "sub_grade", "purpose", "application_type",
             "emp_length", "home_ownership", "verification_status", "addr_state"]

def variant_check(df, cols):
    """Distinct values before vs after normalizing case and spaces."""
    rows = []
    for c in cols:
        raw = df[c].dropna().astype(str)
        rows.append({"column": c, "distinct_raw": raw.nunique(),
                     "distinct_normalized": raw.str.strip().str.lower().nunique()})
    out = pd.DataFrame(rows)
    out["has_variants"] = out["distinct_raw"] > out["distinct_normalized"]
    return out

print("applications"); display(variant_check(apps, TEXT_COLS))
print("holdout"); display(variant_check(hold, TEXT_COLS))

applications


,column,distinct_raw,distinct_normalized,has_variants
0,term,2,2,False
1,grade,7,7,False
2,sub_grade,35,35,False
3,purpose,13,13,False
4,application_type,2,2,False
5,emp_length,11,11,False
6,home_ownership,3,3,False
7,verification_status,3,3,False
8,addr_state,154,61,True


holdout


,column,distinct_raw,distinct_normalized,has_variants
0,term,2,2,False
1,grade,7,7,False
2,sub_grade,35,35,False
3,purpose,12,12,False
4,application_type,2,2,False
5,emp_length,11,11,False
6,home_ownership,3,3,False
7,verification_status,3,3,False
8,addr_state,50,50,False


### Inspect the variants
For each column flagged above, lists every raw spelling grouped under its
normalized value, with counts, to decide how to standardize them.

In [12]:
def show_variants(df, col):
    """Group raw spellings by their normalized form; show groups with more than one spelling."""
    s = df[col].dropna().astype(str)
    groups = s.groupby(s.str.strip().str.lower()).agg(lambda x: x.value_counts().to_dict())
    return groups[groups.apply(len) > 1]

# example: show_variants(hold, "home_ownership")

In [15]:
norm = apps["addr_state"].dropna().astype(str).str.strip().str.lower()
print(norm[~norm.str.fullmatch(r"[a-z]{2}")].value_counts())

addr_state
california        148
texas              93
new york           77
florida            56
georgia            44
pennsylvania       43
new jersey         40
ohio               36
illinois           35
north carolina     32
michigan           24
virginia           24
Name: count, dtype: int64


### Mixed date formats
`issue_d` and `earliest_cr_line` each contain three formats: `Dec-2013` (28,800 rows),
`2013-12-01` (~610) and `12/2013` (~590). Servicing and holdout dates use only the
first. Each format is parsed with its own explicit rule (not pandas' automatic
guessing, which can silently swap day and month), the results are checked for
failures and implausible values, and all dates are stored at month precision.

In [13]:
DATE_FORMATS = {
    r"[A-Za-z]{3}-\d{4}": "%b-%Y",     # Dec-2013
    r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d",  # 2013-12-01
    r"\d{2}/\d{4}": "%m/%Y",           # 12/2013
}

def parse_mixed_dates(s):
    """Parse each known date shape with its own format; anything else becomes NaT."""
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for pattern, fmt in DATE_FORMATS.items():
        mask = s.str.fullmatch(pattern)
        out[mask] = pd.to_datetime(s[mask], format=fmt, errors="coerce")
    return out.dt.to_period("M").dt.to_timestamp()  # month precision

In [14]:
def check_dates(raw, parsed, name):
    """Report unmatched shapes, failed parses, and the parsed range."""
    raw = raw.astype(str).str.strip()
    known = pd.Series(False, index=raw.index)
    for pattern in DATE_FORMATS:
        known |= raw.str.fullmatch(pattern)
    print(f"\n{name}")
    print("  values matching no known format:", int((~known & raw.ne("nan")).sum()))
    print("  matched a format but failed to parse:", int((known & parsed.isna()).sum()))
    print("  range:", parsed.min().date(), "to", parsed.max().date())

    iso = raw[raw.str.fullmatch(r"\d{4}-\d{2}-\d{2}")]
    print("  ISO dates, day part values:", iso.str[-2:].value_counts().to_dict())
    print("  ISO dates, middle part > 12:", int((iso.str[5:7].astype(int) > 12).sum()))

apps["issue_dt"] = parse_mixed_dates(apps["issue_d"])
bureau["earliest_cr_dt"] = parse_mixed_dates(bureau["earliest_cr_line"])
check_dates(apps["issue_d"], apps["issue_dt"], "applications.issue_d")
check_dates(bureau["earliest_cr_line"], bureau["earliest_cr_dt"], "bureau.earliest_cr_line")


applications.issue_d
  values matching no known format: 0
  matched a format but failed to parse: 0
  range: 2013-01-01 to 2015-12-01
  ISO dates, day part values: {'01': 615}
  ISO dates, middle part > 12: 0

bureau.earliest_cr_line
  values matching no known format: 0
  matched a format but failed to parse: 0
  range: 1951-01-01 to 2017-11-01
  ISO dates, day part values: {'01': 607}
  ISO dates, middle part > 12: 0


### Date results and address state
Both date columns parse fully with month precision (no unmatched values, no failed
parses, ISO dates are all day 01, no day/month swaps). However, some `earliest_cr_line`
values fall after the loan was issued, which is impossible and is counted below.
`addr_state` mixes two-letter codes, codes in different cases and full state names;
all are standardized to upper-case two-letter codes.

In [16]:
timeline = apps[["id", "issue_dt"]].merge(bureau[["id", "earliest_cr_dt"]], on="id")
after_issue = timeline["earliest_cr_dt"] > timeline["issue_dt"]
print("earliest_cr_line after issue_d:", int(after_issue.sum()))
display(timeline[after_issue].head())

earliest_cr_line after issue_d: 300


,id,issue_dt,earliest_cr_dt
27,7d1f6edf3c,2013-02-01,2013-11-01
51,13d470c317,2015-05-01,2016-12-01
78,2ba0ab5410,2015-02-01,2017-01-01
86,68732d8749,2015-06-01,2016-09-01
108,8acbab3c06,2013-01-01,2013-04-01


In [17]:
STATE_NAMES = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "district of columbia": "DC", "florida": "FL", "georgia": "GA", "hawaii": "HI",
    "idaho": "ID", "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS",
    "missouri": "MO", "montana": "MT", "nebraska": "NE", "nevada": "NV",
    "new hampshire": "NH", "new jersey": "NJ", "new mexico": "NM", "new york": "NY",
    "north carolina": "NC", "north dakota": "ND", "ohio": "OH", "oklahoma": "OK",
    "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY",
}
VALID_CODES = set(STATE_NAMES.values())

def clean_state(s):
    """Map full names to codes and upper-case codes; anything unrecognized becomes NaN."""
    norm = s.astype(str).str.strip().str.lower()
    mapped = norm.map(STATE_NAMES).fillna(norm.str.upper())
    return mapped.where(mapped.isin(VALID_CODES))

raw_state = apps["addr_state"].astype(str).str.strip()
apps["addr_state_clean"] = clean_state(apps["addr_state"])

print("written as full name:", int(raw_state.str.lower().isin(STATE_NAMES).sum()))
print("code not in upper case:", int((raw_state.str.fullmatch(r"[A-Za-z]{2}")
                                      & (raw_state != raw_state.str.upper())).sum()))
print("unrecognized after cleaning:", int(apps["addr_state_clean"].isna().sum()))
print("distinct states after cleaning:", apps["addr_state_clean"].nunique())

written as full name: 652
code not in upper case: 1311
unrecognized after cleaning: 0
distinct states after cleaning: 49


In [21]:
log_issue("applications", "issue_d",
          "Mixed date formats: Dec-2013 (28,800), 2013-12-01 (615), 12/2013 (585)",
          rows=1200, action="fix",
          reason="Parsed each format explicitly to month precision; all parse, no day/month swaps")

log_issue("bureau", "earliest_cr_line",
          "Mixed date formats: Dec-2013 (28,800), 2013-12-01 (607), 12/2013 (593)",
          rows=1200, action="fix",
          reason="Same rule as issue_d; all values parse")

log_issue("applications", "addr_state",
          "State written as full name (652) or code not in upper case (1,311)",
          rows=652 + 1311, action="fix",
          reason="Mapped names to two-letter codes and upper-cased; 0 unrecognized. "
                 "Kept for fairness analysis; likely excluded from the model as a location proxy")

In [22]:
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...


### A1.2 summary
- `issue_d` and `earliest_cr_line`: 1,200 rows each in non-standard formats → parsed
  explicitly to month precision, all valid.
- `addr_state`: 1,963 rows in non-standard form (full names or lower case) →
  standardized to two-letter codes.
- Other categories (term, grade, purpose, emp_length, home_ownership,
  verification_status, application_type) have no case or spacing variants.
- Servicing and holdout dates use a single format.
- Noted for A1.4: 300 loans have `earliest_cr_line` after `issue_d` (impossible).

## A1.3 Impossible values and unit errors
Each numeric field is checked against the range of values that is physically or
contractually possible (e.g. loans of 1,000–40,000 USD, FICO 300–850, no negative
balances or utilization). Values outside these ranges are counted, then investigated
to separate unit errors (e.g. income entered in thousands) from data-entry errors.
Genuine but extreme values are left for A3 (outlier treatment).

In [23]:
train = apps.merge(bureau, on="id", how="inner")

RULES = {   # column: (lowest possible, highest possible); None = no limit
    "loan_amnt": (1000, 40000), "int_rate": (0, 40), "installment": (0.01, None),
    "annual_inc": (1000, None), "dti": (0, 100),
    "fico_range_low": (300, 850), "fico_range_high": (300, 850),
    "revol_util": (0, None), "bc_util": (0, None),
    "revol_bal": (0, None), "tot_cur_bal": (0, None), "avg_cur_bal": (0, None),
    "total_rev_hi_lim": (0, None), "tot_hi_cred_lim": (0, None),
    "inq_last_6mths": (0, None), "delinq_2yrs": (0, None), "pub_rec": (0, None),
    "pub_rec_bankruptcies": (0, None), "open_acc": (0, None), "total_acc": (0, None),
    "mort_acc": (0, None), "acc_open_past_24mths": (0, None),
    "num_tl_op_past_12m": (0, None), "mths_since_last_delinq": (0, None),
}

def range_violations(df, rules):
    """Count values below the lowest or above the highest possible value."""
    rows = []
    for col, (lo, hi) in rules.items():
        if col not in df:
            continue
        s = df[col]
        below = int((s < lo).sum()) if lo is not None else 0
        above = int((s > hi).sum()) if hi is not None else 0
        if below or above:
            rows.append({"column": col, "below_min": below, "above_max": above,
                         "min": s.min(), "max": s.max()})
    return pd.DataFrame(rows)

print("training (applications + bureau)"); display(range_violations(train, RULES))
print("holdout"); display(range_violations(hold, RULES))

training (applications + bureau)


,column,below_min,above_max,min,max
0,loan_amnt,0,280,1000.0,350000.00
1,annual_inc,899,0,11.5,7500000.00
2,dti,0,1,0.0,100.09
3,revol_util,295,0,-101.3,366.60
4,bc_util,290,0,-102.4,156.20


holdout


,column,below_min,above_max,min,max
0,dti,0,2,0.0,126.8


### Investigating out-of-range values
Each violation is tested against independent evidence before deciding a fix:
- `loan_amnt` above 40,000: is the monthly installment consistent with an amount
  ten times smaller?
- `annual_inc` below 1,000: would yearly loan payments exceed yearly income, and does
  ×1,000 bring these incomes in line with the rest?
- `dti` above 100: inspected directly.
- Negative `revol_util` / `bc_util`: does the absolute value match utilization
  recalculated from balance ÷ limit?

In [24]:
train["term_months"] = train["term"].str.extract(r"(\d+)", expand=False).astype(int)

def expected_installment(amount, rate_pct, months):
    """Standard loan payment formula: amount × r / (1 − (1 + r)^−n), r = monthly rate."""
    r = rate_pct / 100 / 12
    return amount * r / (1 - (1 + r) ** (-months))

train["inst_ratio"] = train["installment"] / expected_installment(
    train["loan_amnt"], train["int_rate"], train["term_months"])

big = train["loan_amnt"] > 40000
print("normal rows, installment ÷ expected:"); print(train.loc[~big, "inst_ratio"].describe())
print("\nrows above 40,000, installment ÷ expected:"); print(train.loc[big, "inst_ratio"].describe())
print("\nmost common large amounts:"); print(train.loc[big, "loan_amnt"].value_counts().head(8))

normal rows, installment ÷ expected:
count    29720.000000
mean         0.999383
std          0.023598
min          0.100002
25%          1.000006
50%          1.000012
75%          1.000020
max          1.073606
Name: inst_ratio, dtype: float64

rows above 40,000, installment ÷ expected:
count    2.800000e+02
mean     1.000013e-01
std      9.995546e-07
min      1.000000e-01
25%      1.000006e-01
50%      1.000011e-01
75%      1.000017e-01
max      1.000054e-01
Name: inst_ratio, dtype: float64

most common large amounts:
loan_amnt
100000.0    24
120000.0    18
150000.0    17
200000.0    13
350000.0    11
240000.0    10
50000.0     10
180000.0     8
Name: count, dtype: int64


In [28]:
x10 = train["inst_ratio"].between(0.099, 0.101)
amount_if_fixed = train["loan_amnt"] / 10

print("×10 rows:", int(x10.sum()))
print("  amount > 40,000 (amount clearly wrong):", int((x10 & (train["loan_amnt"] > 40000)).sum()))
print("  amount/10 < 1,000 (installment must be wrong instead):",
      int((x10 & (amount_if_fixed < 1000)).sum()))
print("  undetermined (amount ≤ 40,000 and amount/10 ≥ 1,000):",
      int((x10 & (train["loan_amnt"] <= 40000) & (amount_if_fixed >= 1000)).sum()))
print("\ninstallment in ×10 rows vs others (median):",
      train.loc[x10, "installment"].median(), "vs", train.loc[~x10, "installment"].median())

×10 rows: 300
  amount > 40,000 (amount clearly wrong): 280
  amount/10 < 1,000 (installment must be wrong instead): 0
  undetermined (amount ≤ 40,000 and amount/10 ≥ 1,000): 20

installment in ×10 rows vs others (median): 399.46500000000003 vs 382.545


### Decision: loan amounts recorded ten times too large
300 loans have an installment exactly one-tenth of what their stated amount requires
(installment ÷ expected = 0.100). For 280 the amount exceeds the 40,000 limit; 20 are
within the limit and would be missed by a range check. Their installments are typical
and no loan shows evidence of a wrong installment, so the amount is the corrupted field.
All 300 are divided by 10 and flagged in `loan_amnt_corrected`.

In [29]:
x10 = train["inst_ratio"].between(0.099, 0.101)

train["loan_amnt_corrected"] = x10.astype(int)          # 1 = amount was fixed
train.loc[x10, "loan_amnt"] = train.loc[x10, "loan_amnt"] / 10

print("rows fixed:", int(x10.sum()))
print("loan_amnt range after fix:", train["loan_amnt"].min(), "to", train["loan_amnt"].max())

log_issue("applications", "loan_amnt",
          "Amount recorded 10× too large: 280 above the 40,000 limit + 20 within it",
          rows=int(x10.sum()), action="fix (÷10) + flag",
          reason="Installment is exactly 1/10 of the payment the stated amount requires "
                 "(ratio 0.100); installments are typical and no loan shows a wrong installment, "
                 "so the amount is the corrupted field. 20 errors were within the valid range "
                 "and found only by the installment cross-check")
save_issues()

rows fixed: 300
loan_amnt range after fix: 1000.0 to 35000.0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...


In [25]:
low = train["annual_inc"] < 1000
yearly_payments = train["installment"] * 12

print("low incomes:"); print(train.loc[low, "annual_inc"].describe())
print("\nshare of low rows where yearly payments exceed income (raw):",
      round((yearly_payments[low] > train.loc[low, "annual_inc"]).mean(), 3))
print("median income, normal rows:", train.loc[~low, "annual_inc"].median())
print("median income, low rows × 1,000:", (train.loc[low, "annual_inc"] * 1000).median())
print("median payment share of income, normal rows:",
      round((yearly_payments[~low] / train.loc[~low, "annual_inc"]).median(), 3))
print("median payment share, low rows × 1,000:",
      round((yearly_payments[low] / (train.loc[low, "annual_inc"] * 1000)).median(), 3))

low incomes:
count    899.000000
mean      73.695217
std       43.331007
min       11.500000
25%       45.000000
50%       63.500000
75%       91.500000
max      470.000000
Name: annual_inc, dtype: float64

share of low rows where yearly payments exceed income (raw): 1.0
median income, normal rows: 65000.0
median income, low rows × 1,000: 63500.0
median payment share of income, normal rows: 0.074
median payment share, low rows × 1,000: 0.077


### Income entered in thousands
899 incomes are below 1,000 (median 63.5). For all of them, yearly loan payments would
exceed the stated income, which is impossible. Multiplied by 1,000, their median income
(63,500) and payment share (7.7%) match the other loans (65,000 and 7.4%).
Before fixing, we confirm a clean gap between these values and real incomes, so the
1,000 cut-off cannot catch a genuine income. Values below 1,000 are then ×1,000,
the same rule given in the brief, and applied at scoring time too.

In [30]:
inc = train["annual_inc"]
print("largest income below 1,000:", inc[inc < 1000].max())
print("incomes between 470 and 1,000:", int(inc.between(470, 1000, inclusive="neither").sum()))
print("incomes between 1,000 and 10,000:", int(inc.between(1000, 10000).sum()))
print("smallest incomes of 1,000 or more:", sorted(inc[inc >= 1000].unique())[:5])

largest income below 1,000: 470.0
incomes between 470 and 1,000: 0
incomes between 1,000 and 10,000: 19
smallest incomes of 1,000 or more: [np.float64(7500.0), np.float64(7900.0), np.float64(8000.0), np.float64(8616.0), np.float64(8800.0)]


In [31]:
low = train["annual_inc"] < 1000

train["annual_inc_corrected"] = low.astype(int)          # 1 = income was fixed
train.loc[low, "annual_inc"] = train.loc[low, "annual_inc"] * 1000

print("rows fixed:", int(low.sum()))
print("annual_inc range after fix:", train["annual_inc"].min(), "to", train["annual_inc"].max())

log_issue("applications", "annual_inc",
          "Income entered in thousands (values below 1,000, median 63.5)",
          rows=int(low.sum()), action="fix (×1,000) + flag",
          reason="Raw values make yearly payments exceed income in 100% of these rows; ×1,000 gives "
                 "median 63,500 and payment share 7.7%, matching other loans (65,000, 7.4%). "
                 "Clean gap between these values and real incomes")
save_issues()

rows fixed: 899
annual_inc range after fix: 7500.0 to 7500000.0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...


In [26]:
cols = ["dti", "annual_inc", "installment", "application_type", "verification_status"]
print("dti > 40 — training:", int((train["dti"] > 40).sum()), "| holdout:", int((hold["dti"] > 40).sum()))
display(train.loc[train["dti"] > 40, cols].sort_values("dti", ascending=False).head(10))
display(hold.loc[hold["dti"] > 40, cols].sort_values("dti", ascending=False))

dti > 40 — training: 4 | holdout: 41


,dti,annual_inc,installment,application_type,verification_status
14637,100.09,27000.0,510.05,Joint App,Source Verified
4091,48.91,38500.0,778.06,Joint App,Source Verified
1603,47.31,40000.0,768.19,Joint App,Verified
24422,44.40,30000.0,461.90,Joint App,Verified


,dti,annual_inc,installment,application_type,verification_status
3056,126.80,12000.0,1174.91,Joint App,Not Verified
3402,124.13,12000.0,545.68,Joint App,Source Verified
4524,58.94,20000.0,587.70,Joint App,Source Verified
3762,52.42,25000.0,167.85,Joint App,Verified
1306,51.57,42000.0,178.24,Joint App,Verified
3669,48.60,51000.0,505.34,Joint App,Verified
2286,48.42,38000.0,301.15,Joint App,Verified
1694,48.20,42000.0,596.44,Joint App,Verified
1269,47.43,110000.0,584.94,Individual,Not Verified
3129,47.26,30272.0,186.48,Individual,Source Verified


In [36]:
log_issue("applications", "dti",
          "DTI above 40: 4 rows in training (all joint applications); 41 in the 2016 holdout, "
          "including individual loans; max 100.09 (training) and 126.8 (holdout)",
          rows=4, action="keep (genuine), cap in A3",
          reason="High DTI is plausible for joint applications; individual loans above 40 appear "
                 "only in 2016, suggesting a policy change, reported as drift")
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...


### Negative utilization
`revol_util` (295 rows) and `bc_util` (290 rows) contain negative values, which are
impossible. Since `revol_util` = `revol_bal` ÷ `total_rev_hi_lim` × 100, it is recalculated
from its parts. If the absolute value of the negative rows agrees with the recalculation
as often as normal rows do, the error is only a flipped sign. `bc_util` cannot be
recalculated, so we check whether the same loans are negative in both columns.
Values above 150% are also compared with the recalculation to see if they are genuine.

In [32]:
recomputed = train["revol_bal"] / train["total_rev_hi_lim"] * 100   # inf when the limit is 0
neg_r = train["revol_util"] < 0
neg_b = train["bc_util"] < 0

def close(a, b, tol=5):
    """Share of rows where a and b are within tol percentage points."""
    return round(((a - b).abs() <= tol).mean(), 3)

print("1. normal rows agree with recalculation:",
      close(train.loc[~neg_r, "revol_util"], recomputed[~neg_r]))
print("2. negative rows, minus sign removed, agree:",
      close(train.loc[neg_r, "revol_util"].abs(), recomputed[neg_r]))
print("3. loans negative in both columns:", int((neg_r & neg_b).sum()),
      "| only revol_util:", int((neg_r & ~neg_b).sum()), "| only bc_util:", int((~neg_r & neg_b).sum()))

high = train["revol_util"] > 150
print("\n4. values above 150%:")
display(train.loc[high, ["revol_util", "revol_bal", "total_rev_hi_lim"]].assign(recomputed=recomputed[high]))

1. normal rows agree with recalculation: 0.961
2. negative rows, minus sign removed, agree: 0.973
3. loans negative in both columns: 1 | only revol_util: 294 | only bc_util: 289

4. values above 150%:


,revol_util,revol_bal,total_rev_hi_lim,recomputed
28306,366.6,2566.0,700.0,366.571429


### Results: utilization
- **revol_util:** with the minus sign removed, the 295 negative values agree with
  balance ÷ limit as often as normal rows do (97.3% vs 96.1% within 5 points), so the
  error is only the sign. Fixed with the absolute value and flagged.
- **bc_util:** negative in 290 different loans (only 1 overlaps with revol_util), so it
  needs its own evidence. It cannot be recalculated; instead we check whether the absolute
  values relate to `revol_util` the same way normal values do.
- **366.6%:** one loan, balance 2,566 on a 700 limit, consistent with its fields, so
  genuine; handled as an extreme in A3.

In [33]:
neg_r = train["revol_util"] < 0

train["revol_util_corrected"] = neg_r.astype(int)
train.loc[neg_r, "revol_util"] = train.loc[neg_r, "revol_util"].abs()
print("revol_util rows fixed:", int(neg_r.sum()), "| new min:", train["revol_util"].min())

log_issue("bureau", "revol_util",
          "Negative utilization (sign flipped)",
          rows=int(neg_r.sum()), action="fix (absolute value) + flag",
          reason="With the sign removed, values agree with revol_bal ÷ total_rev_hi_lim as often "
                 "as normal rows (97.3% vs 96.1% within 5 points)")
save_issues()

revol_util rows fixed: 295 | new min: 0.0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."


In [34]:
neg_b = train["bc_util"] < 0
normal = ~neg_b & train["bc_util"].notna()

print("median bc_util, normal rows:", train.loc[normal, "bc_util"].median())
print("median |bc_util|, negative rows:", train.loc[neg_b, "bc_util"].abs().median())
print("correlation with revol_util, normal rows:",
      round(train.loc[normal, "bc_util"].corr(train.loc[normal, "revol_util"]), 3))
print("correlation with revol_util, negative rows as recorded:",
      round(train.loc[neg_b, "bc_util"].corr(train.loc[neg_b, "revol_util"]), 3))
print("correlation with revol_util, negative rows with sign removed:",
      round(train.loc[neg_b, "bc_util"].abs().corr(train.loc[neg_b, "revol_util"]), 3))

median bc_util, normal rows: 67.0
median |bc_util|, negative rows: 68.0
correlation with revol_util, normal rows: 0.834
correlation with revol_util, negative rows as recorded: -0.835
correlation with revol_util, negative rows with sign removed: 0.835


In [35]:
neg_b = train["bc_util"] < 0

train["bc_util_corrected"] = neg_b.astype(int)
train.loc[neg_b, "bc_util"] = train.loc[neg_b, "bc_util"].abs()
print("bc_util rows fixed:", int(neg_b.sum()), "| new min:", train["bc_util"].min())

log_issue("bureau", "bc_util",
          "Negative utilization (sign flipped), in different loans from revol_util",
          rows=int(neg_b.sum()), action="fix (absolute value) + flag",
          reason="Cannot be recalculated; with the sign removed the median (68 vs 67) and the "
                 "correlation with revol_util (0.835 vs 0.834) match normal rows, while the "
                 "recorded values correlate at -0.835")
save_issues()

bc_util rows fixed: 290 | new min: 0.0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...


**bc_util evidence:** with the sign removed, the negative values have the same typical
size as normal values (median 68 vs 67) and the same relationship with `revol_util`
(correlation 0.835 vs 0.834); as recorded they correlate at −0.835, a mirror image.
The only error is the sign, so the absolute value is taken and the rows are flagged.

### A1.3 summary
- `loan_amnt`: 300 amounts recorded ×10 (280 above the limit, 20 within it) → ÷10, flagged.
- `annual_inc`: 899 incomes in thousands → ×1,000, flagged (clean gap 470 / 7,500).
- `revol_util`: 295 sign-flipped values → absolute value, flagged.
- `bc_util`: 290 sign-flipped values (different loans) → absolute value, flagged.
- `dti`: values above 40 are genuine (joint applications; 2016 policy change), kept.
- All other numeric fields are within their possible ranges; the holdout has no
  impossible values apart from high DTI.
- Genuine extremes (income 7.5M, revol_util 366.6%) are left for outlier treatment in A3.

## A1.4 Contradictions within the data
Some fields constrain each other: the upper FICO bound cannot be below the lower one,
bankruptcies cannot exceed public records, accounts opened in 12 months cannot exceed
those opened in 24 months, and so on. Each rule is counted here first; each violation is
then investigated before deciding a fix.

In [37]:
CHECKS = {
    "fico_high < fico_low": train["fico_range_high"] < train["fico_range_low"],
    "bankruptcies > pub_rec": train["pub_rec_bankruptcies"] > train["pub_rec"],
    "opened 12m > opened 24m": train["num_tl_op_past_12m"] > train["acc_open_past_24mths"],
    "open_acc > total_acc": train["open_acc"] > train["total_acc"],
    "revolving limit 0 but balance > 0": (train["total_rev_hi_lim"] == 0) & (train["revol_bal"] > 0),
    "earliest credit line after issue": train["earliest_cr_dt"] > train["issue_dt"],
    "sub_grade not starting with grade": train["sub_grade"].str[0] != train["grade"],
}

print(pd.Series({name: int(mask.sum()) for name, mask in CHECKS.items()}, name="violations"))

fico_high < fico_low                 150
bankruptcies > pub_rec                 0
opened 12m > opened 24m                0
open_acc > total_acc                   0
revolving limit 0 but balance > 0      0
earliest credit line after issue     300
sub_grade not starting with grade      0
Name: violations, dtype: int64


### FICO range: high below low (150 rows)
FICO is reported as a range whose upper bound is normally the lower bound + 4
(e.g. 690–694). If the 150 violating rows have the two values swapped, their
low − high gap should equal the normal high − low gap.

In [38]:
fico_bad = train["fico_range_high"] < train["fico_range_low"]
gap = train["fico_range_high"] - train["fico_range_low"]

print("normal rows, high − low:", gap[~fico_bad].value_counts().to_dict())
print("violating rows, low − high:", (-gap[fico_bad]).value_counts().to_dict())
display(train.loc[fico_bad, ["id", "fico_range_low", "fico_range_high"]].head())

normal rows, high − low: {4.0: 29848, 5.0: 2}
violating rows, low − high: {4.0: 150}


,id,fico_range_low,fico_range_high
10,72cbf954d2,719.0,715.0
158,67ac18044a,709.0,705.0
771,8ba8fb28f5,744.0,740.0
868,ddc13c53ef,669.0,665.0
1320,4959ae2636,704.0,700.0


**Result:** in all 150 rows, low − high is exactly 4, mirroring the normal +4 gap,
so the two values are swapped. Fixed by setting low to the smaller value and high to
the larger; the same rule handles swaps at scoring time.

In [39]:
import numpy as np

fico_bad = train["fico_range_high"] < train["fico_range_low"]
low, high = train["fico_range_low"], train["fico_range_high"]

train["fico_swapped"] = fico_bad.astype(int)
train["fico_range_low"], train["fico_range_high"] = np.minimum(low, high), np.maximum(low, high)
print("rows fixed:", int(fico_bad.sum()),
      "| remaining violations:", int((train["fico_range_high"] < train["fico_range_low"]).sum()))

log_issue("bureau", "fico_range_low / fico_range_high",
          "Upper FICO bound below the lower bound (values swapped)",
          rows=int(fico_bad.sum()), action="fix (swap) + flag",
          reason="In all 150 rows low − high = 4, mirroring the normal high − low = 4 "
                 "(29,848 rows), so the two values were swapped")
save_issues()

rows fixed: 150 | remaining violations: 0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...
9,bureau,fico_range_low / fico_range_high,Upper FICO bound below the lower bound (values...,150,fix (swap) + flag,"In all 150 rows low − high = 4, mirroring the ..."


### First credit line after the loan was issued (300 rows)
A borrower's first credit line must predate the application. We test whether these
dates come from one of the non-standard formats, whether they are shifted by a constant
amount (which could be corrected), and whether the borrowers' account history is
consistent with a long credit history.

In [40]:
bad = train["earliest_cr_dt"] > train["issue_dt"]
raw = train["earliest_cr_line"].astype(str).str.replace(r"[A-Za-z]", "A", regex=True)\
                                            .str.replace(r"\d", "9", regex=True)
months_after = ((train["earliest_cr_dt"] - train["issue_dt"]).dt.days / 30.44).round()

print("raw format of bad rows:", raw[bad].value_counts().to_dict())
print("\nmonths after issue:"); print(months_after[bad].describe())
print("\nyear of earliest credit line (bad rows):")
print(train.loc[bad, "earliest_cr_dt"].dt.year.value_counts().sort_index().to_dict())
print("\nmedian total_acc — bad:", train.loc[bad, "total_acc"].median(),
      "| normal:", train.loc[~bad, "total_acc"].median())

raw format of bad rows: {'AAA-9999': 289, '9999-99-99': 6, '99/9999': 5}

months after issue:
count    300.000000
mean      11.356667
std        6.518519
min        1.000000
25%        6.000000
50%       11.000000
75%       17.000000
max       23.000000
dtype: float64

year of earliest credit line (bad rows):
{2013: 16, 2014: 49, 2015: 97, 2016: 105, 2017: 33}

median total_acc — bad: 24.0 | normal: 24.0


**Result:** the bad dates appear in all three raw formats in the same proportions as the
whole column, so parsing did not cause them. They fall 1–23 months after issue with no
constant shift, so the true date cannot be recovered. These borrowers have a normal
number of accounts (median 24, same as others), so the history is real and the date is
wrong. The value is set to missing and flagged; the rows are kept. Imputation is
decided in A4.

In [41]:
bad = train["earliest_cr_dt"] > train["issue_dt"]

train["earliest_cr_invalid"] = bad.astype(int)
train.loc[bad, "earliest_cr_dt"] = pd.NaT
print("rows set to missing:", int(bad.sum()),
      "| remaining after issue:", int((train["earliest_cr_dt"] > train["issue_dt"]).sum()))

log_issue("bureau", "earliest_cr_line",
          "First credit line dated after the loan was issued (impossible)",
          rows=int(bad.sum()), action="set to missing + flag (impute in A4)",
          reason="Dates fall 1–23 months after issue with no constant shift (not recoverable); "
                 "spread across all raw formats (not a parsing error); borrowers have normal "
                 "account counts (median total_acc 24), so the date is wrong, not the history")
save_issues()

rows set to missing: 300 | remaining after issue: 0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...
9,bureau,fico_range_low / fico_range_high,Upper FICO bound below the lower bound (values...,150,fix (swap) + flag,"In all 150 rows low − high = 4, mirroring the ..."


### Installment mismatches (not ×10)
After fixing the ×10 amounts, the installment ratio is recomputed. For the remaining
mismatches we test which field explains them: does swapping the term (36 ↔ 60) restore
the match, and is the interest rate far from the usual rate for its sub-grade in the
same quarter? Rates are set from the sub-grade, so they provide independent evidence.

In [42]:
train["inst_ratio"] = train["installment"] / expected_installment(
    train["loan_amnt"], train["int_rate"], train["term_months"])
mism = ~train["inst_ratio"].between(0.98, 1.02)
print("mismatches:", int(mism.sum()))
print(train.loc[mism, "inst_ratio"].describe())

other_term = 96 - train["term_months"]                 # 36 → 60, 60 → 36
alt_ratio = train["installment"] / expected_installment(
    train["loan_amnt"], train["int_rate"], other_term)
print("\nexplained by a swapped term:", int((mism & alt_ratio.between(0.98, 1.02)).sum()))

quarter = train["issue_dt"].dt.to_period("Q")
usual_rate = train.groupby(["sub_grade", quarter])["int_rate"].transform("median")
rate_gap = (train["int_rate"] - usual_rate).abs()
print("int_rate > 2 points from its sub-grade's usual rate — mismatches:",
      int((mism & (rate_gap > 2)).sum()), "| other loans:", int((~mism & (rate_gap > 2)).sum()))

display(train.loc[mism, ["loan_amnt", "int_rate", "sub_grade", "term", "installment",
                         "inst_ratio"]].assign(usual_rate=usual_rate[mism]).head(10))

mismatches: 11
count    11.000000
mean      0.924592
std       0.172596
min       0.501917
25%       0.878983
50%       0.947997
75%       1.052450
max       1.073606
Name: inst_ratio, dtype: float64

explained by a swapped term: 0
int_rate > 2 points from its sub-grade's usual rate — mismatches: 8 | other loans: 1


,loan_amnt,int_rate,sub_grade,term,installment,inst_ratio,usual_rate
7581,15000.0,6.00,C3,36 months,476.38,1.043940,13.33
10622,4000.0,6.00,B1,36 months,124.74,1.025083,9.67
11972,15175.0,6.00,D4,60 months,314.13,1.070745,17.57
15293,23075.0,6.00,F5,60 months,414.59,0.929356,25.57
23011,11675.0,6.00,D2,60 months,171.30,0.758937,16.55
23316,16000.0,15.61,C5,60 months,193.63,0.501917,15.61
24866,29325.0,11.67,B4,60 months,613.77,0.947997,11.67
25519,10525.0,6.00,D1,36 months,339.71,1.060961,15.61
25618,12675.0,6.00,D2,60 months,263.08,1.073606,16.29
28601,24000.0,6.00,B3,36 months,654.02,0.895763,9.99


**Result:** after the ×10 fix, only 11 of 30,000 loans have an installment inconsistent
with amount, rate and term (ratio 0.50–1.07). Swapping the term explains none. In 8,
the interest rate is more than 2 points from the usual rate for its sub-grade and
quarter (versus 1 of the other ~29,990 loans), so the rate is likely wrong; 3 remain
unexplained. The correct values cannot be recovered and 11 rows cannot materially
affect the model, so the rows are kept unchanged and flagged.

In [43]:
train["installment_mismatch"] = mism.astype(int)

log_issue("applications", "installment / int_rate",
          "Installment inconsistent with amount, rate and term (ratio 0.50–1.07), not ×10",
          rows=int(mism.sum()), action="flag, keep",
          reason="Term swap explains none; in 8 the rate is >2 points from its sub-grade's usual "
                 "rate (vs 1 of ~29,990 other loans), so the rate is likely wrong, but the true value "
                 "cannot be recovered. 11 rows cannot materially affect the model")
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...
9,bureau,fico_range_low / fico_range_high,Upper FICO bound below the lower bound (values...,150,fix (swap) + flag,"In all 150 rows low − high = 4, mirroring the ..."


### A1.4 summary
- FICO range swapped (150) → fixed, flagged.
- First credit line after issue (300) → set to missing, flagged; imputed in A4.
- Installment inconsistent, not ×10 (11) → likely wrong rate in 8; kept, flagged.
- No violations: bankruptcies ≤ public records, 12-month ≤ 24-month accounts,
  open ≤ total accounts, no revolving balance with a zero limit, sub-grade matches grade.

## A1.5 Leakage and label checks

### bureau_recovery_flag
An extra column, not described in the brief, appears in the bureau file. "Recovery"
refers to money collected after a loan is charged off, so this may be post-default
information stored in an application-time file. We test its default rate, whether it
matches recoveries recorded in the servicing file, and whether it exists in the holdout.

In [44]:
print("values:"); print(train["bureau_recovery_flag"].value_counts(dropna=False))

print("\ndefault rate by flag:")
print(train.groupby("bureau_recovery_flag")["defaulted"].agg(loans="count", default_rate="mean"))

check = train[["id", "bureau_recovery_flag", "defaulted"]].merge(
    serv[["id", "recoveries"]], on="id")
print("\nflag vs servicing recoveries > 0:")
print(pd.crosstab(check["bureau_recovery_flag"], check["recoveries"] > 0,
                  rownames=["flag"], colnames=["recoveries > 0"]))

print("\npresent in holdout:", "bureau_recovery_flag" in hold.columns)

values:
bureau_recovery_flag
0    25793
1     4207
Name: count, dtype: int64

default rate by flag:
                      loans  default_rate
bureau_recovery_flag                     
0                     25793      0.054666
1                      4207      1.000000

flag vs servicing recoveries > 0:
recoveries > 0  False  True 
flag                        
0               25793      0
1                   0   4207

present in holdout: True


**Result: leakage.** Every loan with `bureau_recovery_flag = 1` defaulted (4,207 of 4,207),
against 5.5% of the others, and the flag matches "recoveries > 0" in the servicing file
exactly (no exceptions). It is therefore this loan's post-default recovery status copied
into an application-time file. It exists in the holdout, but cannot exist for a real
applicant at decision time, so it is excluded from all modeling.

In [45]:
log_issue("bureau", "bureau_recovery_flag",
          "Extra column not in the brief; equals 'recoveries > 0' from servicing (post-default)",
          rows=int(train["bureau_recovery_flag"].sum()), action="exclude from model (leakage)",
          reason="Default rate 100% when set (4,207/4,207) vs 5.5% otherwise; matches servicing "
                 "recoveries exactly. Present in holdout but cannot exist at application time")
save_issues()

print("holdout values:"); print(hold["bureau_recovery_flag"].value_counts(dropna=False))

holdout values:
bureau_recovery_flag
0    5000
Name: count, dtype: int64


In the 2016 holdout the flag is 0 for all 5,000 loans, as it would be at real application
time. A model trained with it would learn "flag = 1 → default" and then see only zeros at
scoring time, so it would systematically underestimate risk: a concrete example of
leakage creating a gap between training and real-world performance.

### Label vs servicing
The servicing file is a second record of each loan's outcome. A fully paid loan should
have repaid its full principal, with no recoveries or debt settlement; a defaulted loan
should not have repaid all of its principal; and no payment can predate the loan.
Contradictions are counted in both directions. The principal repaid also provides an
independent check of the ×10 loan-amount correction.

In [46]:
lab = train[["id", "defaulted", "loan_amnt", "issue_dt", "loan_amnt_corrected"]].merge(serv, on="id")
lab["last_pymnt_dt"] = parse_mixed_dates(lab["last_pymnt_d"])
lab["principal_share"] = lab["total_rec_prncp"] / lab["loan_amnt"]

paid, dflt = lab["defaulted"] == 0, lab["defaulted"] == 1

print("fully paid, principal share:"); print(lab.loc[paid, "principal_share"].describe())
print("\nfully paid but repaid < 99% of principal:", int((paid & (lab["principal_share"] < 0.99)).sum()))
print("fully paid but recoveries > 0:", int((paid & (lab["recoveries"] > 0)).sum()))
print("defaulted but repaid ≥ 99% of principal:", int((dflt & (lab["principal_share"] >= 0.99)).sum()))
print("last payment before issue:", int((lab["last_pymnt_dt"] < lab["issue_dt"]).sum()))

print("\ndebt settlement vs label:")
print(pd.crosstab(lab["debt_settlement_flag"], lab["defaulted"], dropna=False))

fixed_paid = paid & (lab["loan_amnt_corrected"] == 1)
print("\n×10-corrected loans that were fully paid, principal share (should be ≈ 1):")
print(lab.loc[fixed_paid, "principal_share"].describe())

fully paid, principal share:
count    24383.000000
mean         1.000000
std          0.000056
min          0.991304
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: principal_share, dtype: float64

fully paid but repaid < 99% of principal: 0
fully paid but recoveries > 0: 0
defaulted but repaid ≥ 99% of principal: 1
last payment before issue: 0

debt settlement vs label:
defaulted                 0     1
debt_settlement_flag             
N                     24383  4854
Y                         0   763

×10-corrected loans that were fully paid, principal share (should be ≈ 1):
count    243.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: principal_share, dtype: float64


**Result: the label is consistent with servicing.** All 24,383 fully paid loans repaid
their full principal (share ≥ 0.991), none have recoveries or debt settlement, and no
payment predates its loan. All 763 settled loans are labeled defaulted. One defaulted
loan repaid ≥ 99% of principal, plausible when interest and fees remain unpaid, so it
is kept. The servicing file also independently confirms the ×10 correction: the 243
corrected loans that were fully paid repaid exactly their corrected amount.

In [47]:
log_issue("servicing vs applications", "defaulted",
          "Label checked against servicing: no contradictions; 1 defaulted loan repaid ≥99% of principal",
          rows=1, action="keep",
          reason="Fully paid loans all repaid full principal, have no recoveries or settlement; "
                 "settled loans all defaulted. The 1 case is plausible (interest/fees unpaid). "
                 "Servicing also confirms the ×10 loan_amnt fix (243 paid loans: principal share = 1.0)")
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...
9,bureau,fico_range_low / fico_range_high,Upper FICO bound below the lower bound (values...,150,fix (swap) + flag,"In all 150 rows low − high = 4, mirroring the ..."


## A1.6 Free-text fields
`emp_title` and `title` are typed freely, so the same value can be written many ways.
Here we measure the problem: distinct values before and after basic normalization
(case, spaces, punctuation), how many titles occur only once, and disguised missing
values such as "n/a" or "none". Grouping job titles into categories is done in A5.

In [48]:
def normalize_text(s):
    """Lower-case, replace punctuation with spaces, and collapse repeated spaces."""
    return (s.astype(str).str.lower()
             .str.replace(r"[^a-z0-9 ]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

emp = train["emp_title"].dropna()
emp_norm = normalize_text(emp)

print("non-missing emp_title:", len(emp))
print("distinct raw:", emp.nunique(), "| distinct after normalizing:", emp_norm.nunique())
counts = emp_norm.value_counts()
print("titles appearing only once:", int((counts == 1).sum()),
      f"({(counts == 1).sum() / len(emp):.0%} of rows)")

DISGUISED = {"n a", "na", "none", "null", "unknown", "not applicable", "self", "", "nan", "0"}
print("disguised missing:", int(emp_norm.isin(DISGUISED).sum()),
      emp_norm[emp_norm.isin(DISGUISED)].value_counts().to_dict())

print("\ntop 20 normalized titles:"); print(counts.head(20).to_string())

non-missing emp_title: 28208
distinct raw: 15657 | distinct after normalizing: 13327
titles appearing only once: 11184 (40% of rows)
disguised missing: 0 {}

top 20 normalized titles:
emp_title
teacher               621
manager               583
owner                 307
supervisor            294
registered nurse      256
sales                 246
driver                227
rn                    225
office manager        188
project manager       173
general manager       169
engineer              142
truck driver          141
director              127
president             123
vice president        118
store manager         116
operations manager    108
sales manager         108
technician            103


### Is `title` redundant with `purpose`?
`title` is a free-text loan name, usually pre-filled from `purpose`. If, for most purposes,
almost all loans carry the same title, the field adds little beyond `purpose` and can be
excluded. It is also never missing in training but missing for 5.5% of the 2016 holdout.

In [49]:
t = normalize_text(train["title"].dropna())
p = train.loc[t.index, "purpose"].str.replace("_", " ")

print("distinct titles (normalized):", t.nunique())
print("title identical to purpose:", f"{(t == p).mean():.0%}")

top_share = (pd.DataFrame({"purpose": p, "title": t})
             .groupby("purpose")["title"]
             .agg(lambda s: s.value_counts(normalize=True).iloc[0]))
print("\nfor each purpose, share of loans with its single most common title:")
print(top_share.sort_values(ascending=False).round(2).to_string())
print("\nmissing — training:", int(train["title"].isna().sum()), "| holdout:", int(hold["title"].isna().sum()))

distinct titles (normalized): 1788
title identical to purpose: 63%

for each purpose, share of loans with its single most common title:
purpose
medical               0.93
small business        0.91
vacation              0.91
home improvement      0.90
major purchase        0.89
other                 0.88
debt consolidation    0.88
car                   0.88
moving                0.87
credit card           0.85
renewable energy      0.85
house                 0.83
wedding               0.30

missing — training: 8 | holdout: 276


**Results.** `emp_title`: 15,657 distinct values (13,327 after normalizing case and
punctuation); 40% of applicants have a title no one else has, and synonyms split the same
job ("registered nurse" / "rn"). It cannot be used as a category directly and is grouped
into job families in A5.
`title`: 63% of titles equal the purpose, and for 12 of 13 purposes 83–93% of loans share
a single title, so it adds little beyond `purpose`. It is also missing for 5.5% of the
holdout vs 8 training rows. It is excluded in favor of `purpose`.

In [50]:
log_issue("applications", "emp_title",
          "Free text: 15,657 distinct values (13,327 after normalizing); 40% of rows unique; synonyms (rn / registered nurse)",
          rows=int(train["emp_title"].notna().sum()), action="normalize + group into job families (A5)",
          reason="Too many unique values to use as a category; case/punctuation variants and synonyms split the same job")

log_issue("applications", "title",
          "Free text largely duplicating purpose: 63% identical; 83–93% share one title per purpose (wedding 30%)",
          rows=int(train["title"].notna().sum()), action="exclude from model (use purpose)",
          reason="Adds little beyond purpose; missing for 276 holdout rows (5.5%) vs 8 in training")
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...
9,bureau,fico_range_low / fico_range_high,Upper FICO bound below the lower bound (values...,150,fix (swap) + flag,"In all 150 rows low − high = 4, mirroring the ..."


### zip_code
`zip_code` holds the first three digits of the US ZIP code (e.g. `941xx`). We check that
every value has this format, and that it agrees with `addr_state`: the first ZIP digit
identifies a group of states (e.g. 9 = CA, OR, WA, AK, HI), so a mismatch means one of
the two location fields is wrong.

In [51]:
ZIP_FIRST_DIGIT = {
    "0": {"CT", "MA", "ME", "NH", "NJ", "RI", "VT"},
    "1": {"DE", "NY", "PA"},
    "2": {"DC", "MD", "NC", "SC", "VA", "WV"},
    "3": {"AL", "FL", "GA", "MS", "TN"},
    "4": {"IN", "KY", "MI", "OH"},
    "5": {"IA", "MN", "MT", "ND", "SD", "WI"},
    "6": {"IL", "KS", "MO", "NE"},
    "7": {"AR", "LA", "OK", "TX"},
    "8": {"AZ", "CO", "ID", "NM", "NV", "UT", "WY"},
    "9": {"AK", "CA", "HI", "OR", "WA"},
}

def zip_state_mismatch(zip_s, state_s):
    """True where the ZIP's first digit doesn't belong to the state's region."""
    first = zip_s.astype(str).str[0]
    return pd.Series([st not in ZIP_FIRST_DIGIT.get(d, set())
                      for d, st in zip(first, state_s)], index=zip_s.index)

for name, df, states in [("training", train, train["addr_state_clean"]),
                         ("holdout", hold, clean_state(hold["addr_state"]))]:
    print(f"\n--- {name} ---")
    print("formats:", pattern_counts(df["zip_code"]).to_dict())
    mism = zip_state_mismatch(df["zip_code"], states)
    print("ZIP region doesn't match state:", int(mism.sum()))
    if mism.any():
        display(df.loc[mism, ["zip_code"]].assign(state=states[mism]).head())


--- training ---
formats: {'999AA': 30000}
ZIP region doesn't match state: 27


,zip_code,state
395,144xx,NE
1210,275xx,CT
2234,070xx,NY
3076,834xx,IN
3527,117xx,NM



--- holdout ---
formats: {'999AA': 5000}
ZIP region doesn't match state: 3


,zip_code,state
321,905xx,GA
4120,820xx,WI
4189,565xx,MI


**Result:** every ZIP has the expected `999xx` format. 27 training loans (0.09%) and 3
holdout loans have a ZIP region that doesn't match the state. The similar rate in the
untouched holdout suggests real-world causes (moves, mailing addresses) rather than
planted errors, and neither field can be shown to be the wrong one. Both are kept and
the rows flagged; location is used only for regional fairness analysis, where
`addr_state` is used.

In [52]:
train["zip_state_mismatch"] = zip_state_mismatch(train["zip_code"], train["addr_state_clean"]).astype(int)

log_issue("applications", "zip_code / addr_state",
          "ZIP first digit inconsistent with state",
          rows=int(train["zip_state_mismatch"].sum()), action="keep + flag",
          reason="Format valid for all rows; similar rate in untouched holdout (3/5,000) suggests real "
                 "causes (moves, mailing address). Cannot tell which field is wrong; location is excluded "
                 "from the model and addr_state is used for regional fairness analysis")
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...
6,bureau,revol_util,Negative utilization (sign flipped),295,fix (absolute value) + flag,"With the sign removed, values agree with revol..."
7,bureau,bc_util,"Negative utilization (sign flipped), in differ...",290,fix (absolute value) + flag,Cannot be recalculated; with the sign removed ...
8,applications,dti,DTI above 40: 4 rows in training (all joint ap...,4,"keep (genuine), cap in A3",High DTI is plausible for joint applications; ...
9,bureau,fico_range_low / fico_range_high,Upper FICO bound below the lower bound (values...,150,fix (swap) + flag,"In all 150 rows low − high = 4, mirroring the ..."


## A1.7 Holdout checks
The 2016 holdout is cleaned automatically by the same rules at scoring time, so we check
which training problems also occur there (×10 amounts, income in thousands, sign-flipped
utilization, swapped FICO, impossible dates), whether it has problems of its own, and
whether it contains category values never seen in training.

In [53]:
h = hold.copy()
h["issue_dt"] = parse_mixed_dates(h["issue_d"])
h["earliest_cr_dt"] = parse_mixed_dates(h["earliest_cr_line"])
h["term_months"] = h["term"].str.extract(r"(\d+)", expand=False).astype(int)
h_ratio = h["installment"] / expected_installment(h["loan_amnt"], h["int_rate"], h["term_months"])

HOLDOUT_CHECKS = {
    "issue year not 2016": h["issue_dt"].dt.year != 2016,
    "earliest credit line after issue": h["earliest_cr_dt"] > h["issue_dt"],
    "FICO high < low": h["fico_range_high"] < h["fico_range_low"],
    "loan amount ×10 (ratio ≈ 0.1)": h_ratio.between(0.099, 0.101),
    "other installment mismatch": ~h_ratio.between(0.098, 0.102) & ~h_ratio.between(0.98, 1.02),
    "income below 1,000": h["annual_inc"] < 1000,
    "negative revol_util or bc_util": (h["revol_util"] < 0) | (h["bc_util"] < 0),
}
print(pd.Series({k: int(v.sum()) for k, v in HOLDOUT_CHECKS.items()}, name="holdout rows"))

print("\ncategories that differ between training and holdout:")
for col in ["term", "grade", "sub_grade", "purpose", "home_ownership",
            "verification_status", "emp_length", "application_type"]:
    new = set(hold[col].dropna()) - set(train[col].dropna())
    gone = set(train[col].dropna()) - set(hold[col].dropna())
    if new or gone:
        print(f"  {col}: new in holdout {new or '-'} | absent from holdout {gone or '-'}")

print("\njoint applications — training:", int((train["application_type"] == "Joint App").sum()),
      "| holdout:", int((hold["application_type"] == "Joint App").sum()))

issue year not 2016                 0
earliest credit line after issue    0
FICO high < low                     0
loan amount ×10 (ratio ≈ 0.1)       0
other installment mismatch          0
income below 1,000                  0
negative revol_util or bc_util      0
Name: holdout rows, dtype: int64

categories that differ between training and holdout:
  purpose: new in holdout - | absent from holdout {'wedding'}

joint applications — training: 16 | holdout: 96


**Result:** none of the training errors (×10 amounts, income in thousands, sign-flipped
utilization, swapped FICO, impossible dates) occur in the holdout; they were confined to
the training files. The cleaning rules stay in the transform as safeguards for future
data. No category appears in the holdout that is unseen in training; `wedding` exists
only in training. Joint applications rise from 16 (0.05%) to 96 (1.9%), a shift the
model cannot learn from 16 examples; they are flagged and reported as drift.

In [54]:
log_issue("holdout", "application_type",
          "Joint applications rise from 16 in training (0.05%) to 96 in holdout (1.9%)",
          rows=96, action="flag; report as drift; recommend manual review",
          reason="16 training examples are too few to learn joint-application risk; they also "
                 "explain most DTI values above 40")
log_issue("applications", "purpose",
          "'wedding' appears in training but not in the 2016 holdout",
          rows=int((train["purpose"] == "wedding").sum()), action="merge into 'other' (A5)",
          reason="Rare level absent from later data; no new categories appear in the holdout")

issues = save_issues()
pd.set_option("display.max_colwidth", 80)
display(issues[["file", "column", "rows", "action"]])

,file,column,rows,action
0,applications,id,600,drop duplicate (keep first)
1,applications,issue_d,1200,fix
2,bureau,earliest_cr_line,1200,fix
3,applications,addr_state,1963,fix
4,applications,loan_amnt,300,fix (÷10) + flag
5,applications,annual_inc,899,"fix (×1,000) + flag"
6,bureau,revol_util,295,fix (absolute value) + flag
7,bureau,bc_util,290,fix (absolute value) + flag
8,applications,dti,4,"keep (genuine), cap in A3"
9,bureau,fico_range_low / fico_range_high,150,fix (swap) + flag


## A1 summary
The four files were audited for duplicates, formats, impossible values, internal and
cross-file contradictions, leakage, free text and holdout differences. Every confirmed
issue is in `reports/audit_issues.csv` with its row count, action and reason.

Key findings:
- Planted errors were confined to the training files and were each proven with
  independent evidence: duplicate ids (600), mixed date formats (1,200 + 1,200),
  loan amounts ×10 (300, confirmed by installments and by principal repaid), income in
  thousands (899), sign-flipped utilization (295 + 290), swapped FICO bounds (150) and
  impossible first-credit dates (300).
- `bureau_recovery_flag` is post-default leakage (100% default when set; identical to
  servicing recoveries) and is excluded from all modeling.
- The label is fully consistent with the servicing records.
- `title` duplicates `purpose` and is excluded; `emp_title` needs grouping (A5).
- The 2016 holdout is free of the planted errors but shows drift: more joint
  applications and individual loans with DTI above 40.